# GreenPlus — YOLO Waste Detector Training

**Runtime → Change runtime type → T4 GPU**

Classes ที่ train:
| Index | ชื่อไทย | English |
|---|---|---|
| 0 | พลาสติก | plastic |
| 1 | กระดาษ | paper |
| 2 | กระดาษลัง | cardboard |
| 3 | เหล็ก | metal |
| 4 | ขวดแก้ว | glass bottle |
| 5 | ขวดน้ำ | plastic bottle |
| 6 | อลูมิเนียม | aluminum can |
| 7 | ไม่ใช่ขยะ | non-recyclable |

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────
!pip install ultralytics roboflow -q
import os
from ultralytics import YOLO
print('✓ ultralytics', YOLO.__module__)

In [ ]:
# ── 2. Roboflow API Key ────────────────────────────────────────────
# สมัครฟรีที่ roboflow.com แล้วกด Settings → API Keys
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # ← ใส่ key ของคุณ

In [ ]:
# ── 3. Download datasets ──────────────────────────────────────────
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset A: garbage-classification-3  (10k+ images: plastic, paper, cardboard, metal, glass)
ds_a = rf.workspace("material-identification") \
         .project("garbage-classification-3") \
         .version(2).download("yolov8", location="/content/ds_a")

# Dataset B: yolov8-trash-detections  (plastic bottle, glass bottle, can, cardboard, paper)
ds_b = rf.workspace("fyp-bfx3h") \
         .project("yolov8-trash-detections") \
         .version(4).download("yolov8", location="/content/ds_b")

print('✓ datasets downloaded')

In [ ]:
# ── 4. Remap + Merge into unified dataset ─────────────────────────
import shutil, glob
from pathlib import Path

# Our final class list (index → Thai label used by greenplus app)
CLASSES = [
    'พลาสติก',    # 0
    'กระดาษ',     # 1
    'กระดาษลัง',  # 2
    'เหล็ก',      # 3
    'ขวดแก้ว',    # 4
    'ขวดน้ำ',     # 5
    'อลูมิเนียม', # 6
    'ไม่ใช่ขยะ',  # 7
]
NC = len(CLASSES)

# Mapping: (dataset, original_class_id) → our class id
# Dataset A classes: 0=plastic 1=paper 2=cardboard 3=metal 4=glass 5=biodegradable
MAP_A = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 7}

# Dataset B classes: 0=battery 1=can 2=cardboard 3=drink_carton 4=glass_bottle
#                    5=paper 6=plastic_bag 7=plastic_bottle 8=bottle_cap 9=pop_tab
MAP_B = {1: 6, 2: 2, 4: 4, 5: 1, 6: 0, 7: 5, 9: 6}
# (battery, drink_carton, bottle_cap skipped — no matching class)

def remap_labels(src_dir, dst_dir, class_map):
    """Copy images + remap label IDs into dst_dir."""
    Path(dst_dir + '/images').mkdir(parents=True, exist_ok=True)
    Path(dst_dir + '/labels').mkdir(parents=True, exist_ok=True)
    for lbl_path in glob.glob(src_dir + '/labels/**/*.txt', recursive=True):
        p = Path(lbl_path)
        lines_out = []
        skip = False
        for line in p.read_text().strip().splitlines():
            parts = line.split()
            if not parts: continue
            orig_id = int(parts[0])
            if orig_id not in class_map:
                continue  # drop this box
            new_id = class_map[orig_id]
            lines_out.append(str(new_id) + ' ' + ' '.join(parts[1:]))
        if not lines_out: continue  # skip images with no valid boxes
        # find matching image
        stem = p.stem
        img_src = None
        for ext in ['.jpg', '.jpeg', '.png', '.webp']:
            candidate = str(p).replace('/labels/', '/images/').replace('.txt', ext)
            if Path(candidate).exists():
                img_src = candidate; break
        if img_src is None: continue
        shutil.copy(img_src, dst_dir + '/images/' + Path(img_src).name)
        (Path(dst_dir + '/labels') / (stem + '.txt')).write_text('\n'.join(lines_out))

for split in ['train', 'valid', 'test']:
    remap_labels(f'/content/ds_a/{split}', f'/content/merged/{split}', MAP_A)
    remap_labels(f'/content/ds_b/{split}', f'/content/merged/{split}', MAP_B)

# Count merged files
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'/content/merged/{split}/images/*'))
    print(f'{split}: {n} images')

In [ ]:
# ── 5. Write data.yaml ────────────────────────────────────────────
import yaml

data_yaml = {
    'path': '/content/merged',
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    NC,
    'names': CLASSES,
}
with open('/content/merged/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print(open('/content/merged/data.yaml').read())

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────
model = YOLO('yolov8n.pt')   # nano: เล็ก เร็ว เหมาะ browser

results = model.train(
    data    = '/content/merged/data.yaml',
    epochs  = 60,
    imgsz   = 640,
    batch   = 16,
    name    = 'greenplus_waste',
    patience= 15,           # early stop ถ้าไม่ improve 15 epochs
    cache   = True,
    device  = 0,            # GPU
    mosaic  = 1.0,
    degrees = 10,
    fliplr  = 0.5,
)
print('mAP50:', results.results_dict.get('metrics/mAP50(B)', 'n/a'))

In [ ]:
# ── 7. Validate ───────────────────────────────────────────────────
best_pt = '/content/runs/detect/greenplus_waste/weights/best.pt'
val_model = YOLO(best_pt)
metrics = val_model.val(data='/content/merged/data.yaml')
print('mAP50   :', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# ── 8. Export to ONNX ─────────────────────────────────────────────
export_model = YOLO(best_pt)
export_model.export(
    format   = 'onnx',
    imgsz    = 640,
    simplify = True,
    opset    = 12,      # onnxruntime-web compatible
)
onnx_path = best_pt.replace('.pt', '.onnx')
print('ONNX saved:', onnx_path)

In [ ]:
# ── 9. Download ───────────────────────────────────────────────────
from google.colab import files
files.download(onnx_path)
print('✓ ดาวน์โหลด best.onnx สำเร็จ')
print()
print('──────────────────────────────────────────')
print('ขั้นตอนต่อไป:')
print('1. เอา best.onnx ใส่ใน  public/model_ai/yolo_stage1.onnx')
print('2. Admin Page → AI Config → YOLO Stage 1 URL:')
print('   /model_ai/yolo_stage1.onnx')
print('3. YOLO Class Labels (เรียงตาม index):')
print('  ', CLASSES)
print('──────────────────────────────────────────')

In [ ]:
# ── (Optional) Test บนรูปตัวอย่าง ─────────────────────────────────
from IPython.display import Image as IPImage
import glob

test_img = glob.glob('/content/merged/test/images/*')[0]
result = val_model.predict(test_img, conf=0.3, save=True, project='/content/preview')
IPImage('/content/preview/predict/' + Path(test_img).name)